# KINDERGARDEN


In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [66]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

##4.01 Importin the OBJ file

In [ ]:
Ground_floor_outer_boundary = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\outer boundary_ground floor.dxf")
Ground_floor_outer_boundary = Wire.ByEdges(Ground_floor_outer_boundary)
print(Wire.IsClosed(Ground_floor_outer_boundary))

print(Wire.IsClosed(Ground_floor_outer_boundary), Wire.IsManifold(Ground_floor_outer_boundary))

Ground_floor_outer_boundary_closed = Wire.Close(Ground_floor_outer_boundary)
print(Wire.IsClosed(Ground_floor_outer_boundary_closed))


print(Wire.IsClosed(Ground_floor_outer_boundary_closed), Wire.IsManifold(Ground_floor_outer_boundary_closed))

ground_floor_inner_boundary = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\inner boundary_ground floor.dxf")
print(ground_floor_inner_boundary)
cluster = Cluster.ByTopologies(ground_floor_inner_boundary)
ground_floor_inner_boundary = Topology.SelfMerge(cluster)


ground_floor_inner_boundary = Topology.Wires(ground_floor_inner_boundary)   

Ground_floor = Face.ByWire(Ground_floor_outer_boundary, ground_floor_inner_boundary)



a lot of claude code to fix the open wire, it turns out there on only one open vertex of the wire

In [ ]:
outer_edges = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\outer boundary_ground floor.dxf")
print(f"Number of edges from DXF: {len(outer_edges)}")

# Build the wire
w = Wire.ByEdges(outer_edges)
print(f"IsClosed: {Wire.IsClosed(w)}")
print(f"Number of edges in wire: {len(Topology.Edges(w))}")
print(f"Number of vertices in wire: {len(Topology.Vertices(w))}")

In [ ]:
outer_edges = Topology.ByDXFPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\outer boundary_ground floor.dxf")
print(f"Edges from DXF: {len(outer_edges)}")

# Use SelfMerge to properly stitch and split into separate wires
cluster = Cluster.ByTopologies(outer_edges)
merged = Topology.SelfMerge(cluster)
wires = Topology.Wires(merged)

print(f"Number of separate wires after SelfMerge: {len(wires)}")
for i, w in enumerate(wires):
    edges_in_wire = len(Topology.Edges(w))
    print(f"  Wire {i}: closed={Wire.IsClosed(w)}, edges={edges_in_wire}")

In [ ]:


# Get all vertices in the wire and count edge connections
all_vertices = Topology.Vertices(w)
print(f"Total vertices: {len(all_vertices)}")

# Find open endpoints (vertices touching only 1 edge)
open_endpoints = []
for v in all_vertices:
    incident_edges = Topology.SuperTopologies(v, w, topologyType="edge")
    if len(incident_edges) == 1:
        open_endpoints.append(v)
        print(f"  Open endpoint at: ({Vertex.X(v):.6f}, {Vertex.Y(v):.6f}, {Vertex.Z(v):.6f})")

print(f"Found {len(open_endpoints)} open endpoints")

In [ ]:
from collections import Counter


all_vertices = Topology.Vertices(w)

# Count incident edges for each vertex
counts = []
for v in all_vertices:
    n = len(Topology.SuperTopologies(v, w, topologyType="edge"))
    counts.append(n)

distribution = Counter(counts)
print(f"Edges-per-vertex distribution: {dict(distribution)}")

# Show any vertex that's not exactly 2-connected
for v in all_vertices:
    n = len(Topology.SuperTopologies(v, w, topologyType="edge"))
    if n != 2:
        print(f"  Vertex with {n} edges at: ({Vertex.X(v):.4f}, {Vertex.Y(v):.4f}, {Vertex.Z(v):.4f})")

In [ ]:


open_v = open_endpoints[0]
all_edges = Topology.Edges(w)

# Find the edge touching the open vertex
stray_edge = None
for e in all_edges:
    for ev in Topology.Vertices(e):
        if Vertex.Distance(ev, open_v) < 0.001:
            stray_edge = e
            break
    if stray_edge:
        break

# Rebuild without it
clean_edges = [e for e in all_edges if e != stray_edge]
print(f"Edges before: {len(all_edges)}, after: {len(clean_edges)}")

cluster = Cluster.ByTopologies(clean_edges)
merged = Topology.SelfMerge(cluster)
fixed_wires = Topology.Wires(merged)
Ground_floor_outer_boundary = fixed_wires[0]
print("Closed now?", Wire.IsClosed(Ground_floor_outer_boundary))

building the ground floor face

In [ ]:
Ground_floor = Face.ByWire(Ground_floor_outer_boundary, ground_floor_inner_boundary)

## 6. Show the geometry

In [ ]:
Topology.Show(Ground_floor,
              faceColor=[210,210,250],
              faceOpacity=0.5,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer = renderer)

creating the grid overlay

In [ ]:
b_r = Wire.BoundingRectangle(Ground_floor)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+100,100))
vRange = list(range(0,int(length)+5000,100))

grid = Grid.EdgesByDistances(Ground_floor, clip=True, uRange=uRange, vRange=vRange)

print("width:", width)
print("length:", length)
print("uRange:", uRange)
print("vRange:", vRange)

showing the geometry and the grid

In [ ]:
Topology.Show(Ground_floor, grid,
              camera=[0,0,20],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

Slicing the floor plan with the grid

In [ ]:
shell = Topology.Slice(Ground_floor, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

showing the resulting shell

In [ ]:
Topology.Show(shell,
              camera=[0,0,20],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

navigation and analysis graphs from the shell

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)

showing the analysis graph

In [ ]:
Topology.Show(analysis_graph,
              camera=[0,0,20],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

shortest path

In [ ]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=Ground_floor)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "blue"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
length = Wire.Length(straight_path)
for edge in edges:
    mid_v = Edge.VertexByParameter(e, 0.5)
    d = Dictionary.ByKeysValues(["width", "color","label"], [7, "red","str(length)"])
    edge = Topology.SetDictionary(edge, d, mid_v)
edges = Topology.Edges(straight_path)
    

show the shortest path

In [ ]:
Topology.Show(Ground_floor, shortest_path, straight_path,
              camera=[0,0,20],
              faceColor=[210,210,250],
              faceOpacity=0.2,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer,
              showVertices=True,
              vertexLabelKey="label",
              showVertexLabel=True,
              )


Closeness centrality

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")


In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,20],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Betweenness centrality

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,20],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Degree centrality

adjacency graph first

In [61]:

kindergarden = Topology.ByOBJPath(r"C:\Users\DD\Desktop\IAAC\Graph ML\Homework\02\assets\ground floor_room_slabs.obj")
print(kindergarden)
faces = Cluster.Faces(kindergarden[0])
print("Number of faces:", len(faces))




Number of faces: 538


new shell from the new faces

In [68]:
new_shell = Shell.ByFaces(faces)
face = Topology.RemoveCoplanarFaces(shell)

In [70]:
Topology.Show(new_shell,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              camera=[0,0,20],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

In [71]:
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 12])
    v = Topology.SetDictionary(v, d)

In [72]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0,0,20],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)